# PAUL Open Model — Google Colab Hardware & Environment Setup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/foundrypaul-cloud/paul-open/blob/main/notebooks/01_colab_environment_setup.ipynb)

This notebook sets up and verifies a reproducible environment for **Google Gemma 4** research on Google Colab.

### Hardware & Target Strategy
- **Tesla T4 (14.56 GiB VRAM)**: Safe development model is **Gemma 4 E4B IT** (~6 GB QLoRA training footprint). **Gemma 4 12B IT** is an experimental target.
- **L4 / RTX 4090 / A5000 (24 GB VRAM)**: **Gemma 4 12B IT** (Unified) and **Gemma 4 26B A4B IT** (MoE Primary Target).
- **A100 / H100 (40/80 GB VRAM)**: **Gemma 4 26B A4B IT** and **Gemma 4 31B IT** (Maximum Dense Capability).

> **Strict Safety Notice**: This setup script verifies environment dependencies and hardware capabilities **without downloading model weights or datasets** and **without starting training**.

## Step 1: Repository Setup & Explicit Package Installation
Clone the repository (if in Colab) and pin verified, compatible versions of all core dependencies.

In [ ]:
import os
import sys

# In Google Colab, clone the repository to access local configs and src package
if "google.colab" in sys.modules or os.environ.get("COLAB_GPU") is not None:
    if not os.path.exists("paul-open"):
        !git clone -q https://github.com/foundrypaul-cloud/paul-open.git
        %cd paul-open
    elif os.path.basename(os.getcwd()) != "paul-open":
        %cd paul-open

# Install explicit, pinned versions of the Gemma 4 ML stack
# Google's Gemma 4 documentation requires Transformers >=5.10.1
!pip install -q --no-cache-dir \
    "torch>=2.11.0" \
    "transformers>=5.13.1" \
    "peft>=0.19.0" \
    "trl>=1.9.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.2.0" \
    "huggingface_hub>=0.28.0" \
    "tensorboard>=2.18.0" \
    "pyyaml>=6.0" \
    "rich>=13.0.0" \
    "sentencepiece>=0.2.0" \
    "tokenizers>=0.21.0"

## Step 2: Environment, CUDA & GPU VRAM Verification

In [ ]:
import platform
import accelerate
import bitsandbytes as bnb
import datasets
import huggingface_hub
import peft
import torch
import transformers
import trl

print("=" * 70)
print(" PAUL OPEN MODEL — COLAB ENVIRONMENT VERIFICATION REPORT")
print("=" * 70)
print(f" Python Version   : {platform.python_version()} (Expected: >=3.12)")
print(f" OS Platform      : {platform.platform()}")
print(f" CUDA Available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f" GPU Device Count : {device_count}")
    for i in range(device_count):
        prop = torch.cuda.get_device_properties(i)
        total_gb = prop.total_memory / (1024 ** 3)
        usable_gb = total_gb * 0.95
        print(f"   [GPU {i}] {prop.name}")
        print(f"          Compute Capability : {prop.major}.{prop.minor}")
        print(f"          Total VRAM         : {total_gb:.2f} GiB")
        print(f"          Usable Est. VRAM   : {usable_gb:.2f} GiB")
        print(f"          CUDA Runtime       : {torch.version.cuda}")
else:
    print(" WARNING: No CUDA GPU detected! Please select Runtime -> Change runtime type -> T4/L4/A100 GPU.")

print("-" * 70)
print(" Pinned Dependency Audit:")
print(f"   - torch             : {torch.__version__} (>=2.11.0 required)")
print(f"   - transformers      : {transformers.__version__} (>=5.10.1 required for Gemma 4)")
print(f"   - peft              : {peft.__version__} (>=0.19.0 required for Gemma 4 QLoRA)")
print(f"   - trl               : {trl.__version__} (>=1.9.0 required)")
print(f"   - bitsandbytes      : {bnb.__version__} (>=0.45.0 required)")
print(f"   - accelerate        : {accelerate.__version__} (>=1.2.0 required)")
print(f"   - datasets          : {datasets.__version__} (>=3.2.0 required)")
print(f"   - huggingface_hub   : {huggingface_hub.__version__} (>=0.28.0 required)")
print("=" * 70)

## Step 3: Configurable Model Strategy & VRAM Budgeting
Select the active model configuration based on the detected hardware.

In [ ]:
# Supported Gemma 4 model targets
TARGET_MODELS = {
    "primary": {
        "model_id": "google/gemma-4-26B-A4B-it",
        "role": "Primary Research Target (MoE)",
        "params": "26B total (~4B active)",
        "arch": "gemma4",
        "min_vram_gb": 22.0,
        "notes": "Flagship target for >=24 GB GPU runtimes (L4 / RTX 4090 / A100).",
    },
    "development": {
        "model_id": "google/gemma-4-12B-it",
        "role": "Development / Fallback Target",
        "params": "12B dense (June 2026 Unified release)",
        "arch": "gemma4_unified",
        "min_vram_gb": 14.0,
        "notes": "Experimental target on T4; comfortable on >=16 GB GPUs.",
    },
    "safe_t4": {
        "model_id": "google/gemma-4-E4B-it",
        "role": "Safe T4 Development Model",
        "params": "4.5B dense (edge-optimized)",
        "arch": "gemma4",
        "min_vram_gb": 6.0,
        "notes": "Recommended safe default for Tesla T4 (14.56 GiB VRAM).",
    },
    "maximum": {
        "model_id": "google/gemma-4-31B-it",
        "role": "Maximum Capability Ceiling",
        "params": "31B dense",
        "arch": "gemma4",
        "min_vram_gb": 28.0,
        "notes": "High-complexity reasoning; requires A100/H100 or multi-GPU.",
    },
}

# Automatically select recommendation based on detected VRAM
if torch.cuda.is_available():
    usable_vram = (torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)) * 0.95
    if usable_vram <= 15.5:
        SELECTED_CONFIG = "safe_t4"  # Safe development model on Tesla T4
    elif usable_vram < 24.0:
        SELECTED_CONFIG = "development"
    elif usable_vram < 48.0:
        SELECTED_CONFIG = "primary"
    else:
        SELECTED_CONFIG = "maximum"
else:
    SELECTED_CONFIG = "safe_t4"

active_spec = TARGET_MODELS[SELECTED_CONFIG]
print(f"Active Selected Tier : {SELECTED_CONFIG.upper()}")
print(f"Target Model ID      : {active_spec['model_id']}")
print(f"Role                 : {active_spec['role']}")
print(f"Parameters           : {active_spec['params']}")
print(f"Architecture Tag     : {active_spec['arch']}")
print(f"Notes                : {active_spec['notes']}")

## Step 4: Transformers Gemma 4 API Configuration Check
Verify that the required training arguments and multimodal flags are supported without loading weights.

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from trl import SFTConfig

# 1. Verify 4-bit QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)
print("✓ BitsAndBytes 4-bit QLoRA configuration initialized successfully.")

# 2. Verify Gemma 4 SFTConfig requirements
# Critical rule: remove_unused_columns=False is required to preserve mm_token_type_ids
sft_config = SFTConfig(
    output_dir="./checkpoints_test",
    remove_unused_columns=False,  # Essential for Gemma 4 multimodal metadata
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_seq_length=2048 if SELECTED_CONFIG == "safe_t4" else 4096,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    report_to="none",  # TensorBoard / W&B configured separately
)
print("✓ SFTConfig for Gemma 4 training initialized successfully.")
print("✓ All pre-flight environment checks complete.")
print("=" * 70)
print(" READY: Environment verified. No weights or datasets were downloaded.")
print("=" * 70)